In [2]:
# Bibliotecas para coletar os dados na PokéAPI (HTTP) e montar o dataset
import requests
import pandas as pd
import time

In [3]:
def fetch_with_retry(url, retries=5, wait=3):
    """Faz GET na URL, repetindo até `retries` vezes com pausa de `wait`s entre tentativas.

    Retorna o JSON da resposta (HTTP 200) ou None se todas as tentativas falharem.
    """
    for attempt in range(retries):
        r = requests.get(url)
        if r.status_code == 200:
            return r.json()
        print(f"\n  HTTP {r.status_code} em {url} (tentativa {attempt + 1}/{retries}), aguardando {wait}s...")
        time.sleep(wait)
    return None


def fetch_all_pokemon_ids():
    """Retorna a lista de IDs de todas as espécies de Pokémon disponíveis na PokéAPI."""
    response = fetch_with_retry("https://pokeapi.co/api/v2/pokemon-species?limit=10000")
    return [int(s["url"].split("/")[-2]) for s in response["results"]]


def fetch_pokemon(pokemon_id):
    """Coleta os dados de um Pokémon na PokéAPI e retorna um dict (ou None se falhar).

    Combina três endpoints: `pokemon` (tipos, altura, peso), `pokemon-species`
    (cor, habitat, forma, geração) e `evolution-chain` (cadeia de evolução).
    """
    poke = fetch_with_retry(f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}")
    if not poke:
        return None

    species = fetch_with_retry(f"https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}")
    if not species:
        return None

    evo_chain = fetch_with_retry(species["evolution_chain"]["url"])

    return {
        "id": pokemon_id,
        "name": poke["name"],
        "types": [t["type"]["name"] for t in poke["types"]],
        "height": poke["height"],
        "weight": poke["weight"],
        "color": species["color"]["name"] if species.get("color") else None,
        "habitat": species["habitat"]["name"] if species.get("habitat") else None,
        "shape": species["shape"]["name"] if species.get("shape") else None,
        "generation": species["generation"]["name"],
        "evolution_chain": evo_chain["chain"] if evo_chain else None,
    }


# Coleta cada Pokémon e salva o dataset bruto em data/pokemon_raw.csv
ids = fetch_all_pokemon_ids()
print(f"{len(ids)} Pokémons encontrados.")

pokemons = []
for i, pokemon_id in enumerate(ids, 1):
    print(f"Buscando {i}/{len(ids)} (#{pokemon_id})...", end="\r")
    result = fetch_pokemon(pokemon_id)
    if result:
        pokemons.append(result)
    else:
        print(f"\n#{pokemon_id} falhou após todas as tentativas.")

df = pd.DataFrame(pokemons)
df.to_csv("../data/pokemon_raw.csv", index=False)
print(f"\nDataset salvo! {len(df)} Pokémons coletados.")
df.head()

1025 Pokémons encontrados.
Buscando 1025/1025 (#1025)...
Dataset salvo! 1025 Pokémons coletados.


,id,name,types,height,weight,color,habitat,shape,generation,evolution_chain
0,1,bulbasaur,"[grass, poison]",7,69,green,grassland,quadruped,generation-i,"{'evolution_details': [], 'evolves_to': [{'evo..."
1,2,ivysaur,"[grass, poison]",10,130,green,grassland,quadruped,generation-i,"{'evolution_details': [], 'evolves_to': [{'evo..."
2,3,venusaur,"[grass, poison]",20,1000,green,grassland,quadruped,generation-i,"{'evolution_details': [], 'evolves_to': [{'evo..."
3,4,charmander,[fire],6,85,red,mountain,upright,generation-i,"{'evolution_details': [], 'evolves_to': [{'evo..."
4,5,charmeleon,[fire],11,190,red,mountain,upright,generation-i,"{'evolution_details': [], 'evolves_to': [{'evo..."
